In [ ]:
# 📌 Install required libraries (if not already installed)
!pip install yfinance beautifulsoup4 lxml --quiet

# 📌 Import libraries
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import time
from tqdm.notebook import tqdm

# 📌 Step X: Get S&P X tickers from Wikipedia
def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    df = pd.read_html(str(table))[0]
    tickers = df['Symbol'].tolist()
    tickers = [ticker.replace('.', '-') for ticker in tickers]  # Fix BRK.B to BRK-B, etc.
    return tickers

# 📌 Step X: Get Average Implied Volatility (IV) for a stock
def get_avg_iv(ticker):
    try:
        stock = yf.Ticker(ticker)
        expirations = stock.options
        if not expirations:
            return None
        nearest_exp = expirations[0]
        options = stock.option_chain(nearest_exp)
        calls = options.calls
        puts = options.puts

        # Get current stock price
        price = stock.history(period="1d")['Close'].iloc[-1]
        atm_calls = calls[(calls['strike'] - price).abs() < 1]
        atm_puts = puts[(puts['strike'] - price).abs() < 1]

        ivs = pd.concat([atm_calls['impliedVolatility'], atm_puts['impliedVolatility']])
        return ivs.mean()
    except Exception as e:
        print(f"{ticker}: {e}")
        return None

# 📌 Step X: Loop through tickers and fetch IV
tickers = get_sp500_tickers()
iv_results = []

print("Fetching Implied Volatility for S&P 500 companies...")

for ticker in tqdm(tickers):
    avg_iv = get_avg_iv(ticker)
    iv_results.append({'Ticker': ticker, 'Average_IV': avg_iv})
    time.sleep(0.5)  # prevent getting blocked

# 📌 Step X: Display and save
df_iv = pd.DataFrame(iv_results)
df_iv.dropna(inplace=True)
df_iv.sort_values(by='Average_IV', ascending=False, inplace=True)

# Display top X
df_iv.head(10)
